# Crypto.Net · 07 — Indexing events into a database

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./07-indexer.id.ipynb)

What you will do: decode ERC-20 `Transfer` logs from Sepolia, run the `EvmLogIndexer` from a checkpoint into SQLite with Entity Framework Core, query the result with LINQ, and crawl Polkadot Asset Hub blocks with `BlockCrawler`. Everything is read-only.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"
#r "nuget: Crypto.Net.Indexer.EntityFrameworkCore, 1.2.0"
#r "nuget: Microsoft.EntityFrameworkCore.Sqlite, 10.0.12"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Evm;
using Crypto.Net.Indexer;
using Crypto.Net.Indexer.EntityFrameworkCore;
using Crypto.Net.Polkadot;
using Microsoft.Data.Sqlite;
using Microsoft.EntityFrameworkCore;
using Microsoft.EntityFrameworkCore.Infrastructure;

const string Usdc = "0x1c7D4B196Cb0C7B01d743Fbc6116a902379C7238";   // Circle USDC on Sepolia
var sepolia = new EvmRpcClient(EvmChain.Sepolia);
ulong head = await sepolia.GetBlockNumberAsync();
$"Sepolia head {head:N0}"

## Decoding events

`EvmEvent.Parse` reads a human-readable Solidity event. `Decode` turns a log into named values: indexed parameters come from the topics, the rest from the data.

In [ ]:
var transfer = EvmEvent.Parse("event Transfer(address indexed from, address indexed to, uint256 value)");
Console.WriteLine($"{transfer.Signature} → topic0 {transfer.Topic0[..18]}…");

var logs = await sepolia.GetLogsAsync(new EvmLogFilter(head - 100, head) { Addresses = [Usdc], Topics = [[transfer.Topic0]] });
foreach (var log in logs.Take(5))
{
    var e = transfer.Decode(log);
    Console.WriteLine($"block {log.BlockNumber}: {e["from"]} → {e["to"]}  {(decimal)(BigInteger)e["value"]! / 1_000_000m} USDC");
}
$"{logs.Count} transfers in the last 100 blocks"

## Indexing into SQLite

The indexer pages `eth_getLogs` from its checkpoint to the head minus `Confirmations`. When a provider refuses a range, it halves the range and retries. After each batch it writes a checkpoint, so a restart resumes where it stopped. `EfEvmLogStore` writes raw logs and ERC-20 transfers. Its inserts are idempotent, so a redelivered batch does no harm.

In [ ]:
var connection = new SqliteConnection("Data Source=:memory:");
await connection.OpenAsync();
var factory = new PooledDbContextFactory<CryptoNetIndexerDbContext>(
    new DbContextOptionsBuilder<CryptoNetIndexerDbContext>().UseSqlite(connection).Options);
await using (var db = await factory.CreateDbContextAsync()) await db.Database.EnsureCreatedAsync();

var checkpoints = new EfCheckpointStore<CryptoNetIndexerDbContext>(factory);
var store = new EfEvmLogStore<CryptoNetIndexerDbContext>(factory, new EvmLogStoreOptions { ChainId = "sepolia" });
var indexer = new EvmLogIndexer(sepolia, checkpoints, store, new EvmLogIndexerOptions
{
    Name = "usdc",
    Addresses = [Usdc],
    Events = [EvmEvent.Erc20Transfer],
    StartBlock = head - 500,
    Confirmations = 5,
    MaxBlockRange = 100,
});
ulong? last = await indexer.RunOnceAsync();
$"indexed up to block {last:N0}; checkpoint = {await checkpoints.GetAsync("usdc"):N0}"

In [ ]:
var query = await factory.CreateDbContextAsync();
var top = (await query.TokenTransfers.Where(t => t.ChainId == "sepolia").ToListAsync())
    .GroupBy(t => t.To)
    .Select(g => new { Recipient = g.Key, Transfers = g.Count(), Usdc = g.Sum(t => (decimal)t.RawValue) / 1_000_000m })
    .OrderByDescending(x => x.Usdc)
    .Take(5)
    .ToList();
await query.DisposeAsync();
top

Running `RunOnceAsync` again only processes new blocks, because the checkpoint remembers where it stopped. In an app, register the same pipeline with `AddEvmLogIndexerToDatabase` and it runs as a hosted service. The `cnet-worker` template does exactly that.

## Any chain: `BlockCrawler`

The crawler calls your handler once for every height, in order, with checkpoints. Here it counts the events in Polkadot Asset Hub blocks.

In [ ]:
var dot = new PolkadotRpcClient(PolkadotChain.PolkadotAssetHub);
ulong dotHead = await dot.GetBlockNumberAsync();
var counts = new List<string>();
var crawler = new BlockCrawler(dot, new InMemoryCheckpointStore(), async (height, ct) =>
{
    var events = await dot.GetEventsAsync(await dot.GetBlockHashAsync(height, ct), ct);
    counts.Add($"#{height}: {events.Count} events, {events.Count(e => e.Is("Balances", "Transfer"))} transfers");
}, new BlockCrawlerOptions { StartBlock = dotHead - 4, Confirmations = 0, MaxBlocksPerPoll = 5 });
await crawler.RunOnceAsync();
string.Join("\n", counts)

In [ ]:
sepolia.Dispose(); dot.Dispose(); connection.Dispose();
"disposed"